# EEEM068: Industrial Waste Classification
## Notebook 04 — Results Analysis

**Author:** Scott Lewis  
**Date:** March 2026  

---

### Objectives

This notebook aggregates and compares results across all models and runs:

1. Load all `test_metrics.json` files from completed runs
2. Compare models across all metrics (Accuracy, F1, AUC, mAP)
3. Overlay training curves for cross-model comparison
4. Analyse Phase 1 grid results (lr × batch size)
5. Identify best configurations for Phase 2
6. Per-class difficulty analysis across models

---

> **Note:** Run this notebook after completing Phase 1 experiments.
> Results will populate as more runs complete.

## 0. Setup

In [ ]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

# paths — results stored relative to project root
PROJECT_ROOT = Path('../../')
RESULTS_ROOT = PROJECT_ROOT / 'experiments' / 'results'
DATASET_DIR  = Path('dataset')

MODELS = ['resnet50', 'efficientnet_b3', 'swin_t', 'convnext_t']

# class mapping
with open(DATASET_DIR / 'class_mapping.json') as f:
    mapping = json.load(f)
label_to_class = {int(k): v for k, v in mapping['label_to_class'].items()}
n_classes      = mapping['n_classes']
class_names    = [label_to_class[i] for i in range(n_classes)]

print('Setup complete')
print(f'Results root: {RESULTS_ROOT}')

## 1. Load All Results

In [ ]:
def load_all_results(results_root: Path) -> pd.DataFrame:
    """
    Walk the results directory and load all test_metrics.json files.
    Returns a DataFrame with one row per completed run.
    """
    records = []

    for model_dir in sorted(results_root.iterdir()):
        if not model_dir.is_dir():
            continue
        model_name = model_dir.name

        for run_dir in sorted(model_dir.iterdir()):
            if not run_dir.is_dir():
                continue

            metrics_path = run_dir / 'test_metrics.json'
            config_path  = run_dir / 'config.yaml'

            if not metrics_path.exists():
                continue

            with open(metrics_path) as f:
                metrics = json.load(f)

            # try to load config for hyperparameters
            config = {}
            if config_path.exists():
                import yaml
                with open(config_path) as f:
                    config = yaml.safe_load(f)

            record = {
                'model':      model_name,
                'run':        run_dir.name,
                'lr':         config.get('training', {}).get('learning_rate', None),
                'batch_size': config.get('training', {}).get('batch_size', None),
                'phase':      config.get('run', {}).get('phase', None),
                'accuracy':   metrics.get('accuracy'),
                'f1_macro':   metrics.get('f1_macro'),
                'f1_weighted':metrics.get('f1_weighted'),
                'precision':  metrics.get('precision_macro'),
                'recall':     metrics.get('recall_macro'),
                'auc':        metrics.get('auc'),
                'mAP':        metrics.get('mAP'),
            }
            records.append(record)

    return pd.DataFrame(records)


results_df = load_all_results(RESULTS_ROOT)

if results_df.empty:
    print('No completed runs found yet. Run experiments first.')
else:
    print(f'Found {len(results_df)} completed runs:')
    print(results_df[['model','run','lr','batch_size','accuracy','f1_macro','auc']].to_string(index=False))

## 2. Model Comparison — Best Run Per Model

In [ ]:
if not results_df.empty:
    # best run per model (highest F1 macro)
    best_per_model = results_df.loc[results_df.groupby('model')['f1_macro'].idxmax()]
    best_per_model = best_per_model.sort_values('f1_macro', ascending=False)

    print('Best run per model:')
    print(best_per_model[['model','run','accuracy','f1_macro','precision',
                           'recall','auc','mAP']].round(4).to_string(index=False))
else:
    print('No results available yet.')

In [ ]:
if not results_df.empty and len(best_per_model) > 0:
    metrics_to_plot = ['accuracy', 'f1_macro', 'precision', 'recall', 'auc', 'mAP']

    x      = np.arange(len(metrics_to_plot))
    width  = 0.8 / max(len(best_per_model), 1)
    colours = ['#7570b3', '#1b9e77', '#d95f02', '#e7298a']

    fig, ax = plt.subplots(figsize=(13, 5))

    for i, (_, row) in enumerate(best_per_model.iterrows()):
        vals = [row.get(m, 0) or 0 for m in metrics_to_plot]
        offset = (i - len(best_per_model) / 2 + 0.5) * width
        bars = ax.bar(x + offset, vals, width * 0.9,
                      label=row['model'], color=colours[i % len(colours)],
                      edgecolor='white')

    ax.set_xticks(x)
    ax.set_xticklabels([m.replace('_', ' ').title() for m in metrics_to_plot])
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Score')
    ax.set_title('Model Comparison — Best Run per Model (all metrics)')
    ax.legend()
    plt.tight_layout()
    plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: model_comparison.png')
else:
    print('Waiting for results — re-run after experiments complete.')

## 3. Phase 1 Grid Heatmaps

Visualising how learning rate and batch size interact for each model.

In [ ]:
if not results_df.empty:
    phase1_df = results_df[results_df['phase'] == 1].copy()

    for model in MODELS:
        model_df = phase1_df[phase1_df['model'] == model]
        if model_df.empty:
            print(f'{model}: no phase 1 results yet')
            continue

        pivot = model_df.pivot_table(
            index='lr', columns='batch_size', values='f1_macro'
        )

        fig, ax = plt.subplots(figsize=(6, 4))
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd',
                    ax=ax, vmin=0, vmax=1)
        ax.set_title(f'{model} — Phase 1 F1 Grid (lr × batch size)')
        ax.set_xlabel('Batch size')
        ax.set_ylabel('Learning rate')
        plt.tight_layout()
        plt.savefig(f'phase1_grid_{model}.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f'Saved: phase1_grid_{model}.png')
else:
    print('No phase 1 results yet.')

## 4. Training Curves Comparison

Overlaying validation F1 curves across all models for direct comparison.

In [ ]:
def load_training_history(results_root: Path, model: str, run: str) -> dict:
    """Load metrics.json (training history) for a specific run."""
    path = results_root / model / run / 'metrics.json'
    if not path.exists():
        return {}
    with open(path) as f:
        return json.load(f)


if not results_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    colours   = ['#7570b3', '#1b9e77', '#d95f02', '#e7298a']

    for i, (_, row) in enumerate(best_per_model.iterrows()):
        history = load_training_history(RESULTS_ROOT, row['model'], row['run'])
        if not history or 'val_f1' not in history:
            continue

        epochs = range(1, len(history['val_f1']) + 1)
        colour = colours[i % len(colours)]

        axes[0].plot(epochs, history['val_loss'], label=row['model'], color=colour)
        axes[1].plot(epochs, history['val_f1'],   label=row['model'], color=colour)

    axes[0].set_title('Validation Loss — all models')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()

    axes[1].set_title('Validation F1 (macro) — all models')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('F1')
    axes[1].legend()

    plt.suptitle('Cross-model training curve comparison', y=1.02)
    plt.tight_layout()
    plt.savefig('training_curves_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: training_curves_comparison.png')
else:
    print('No results yet — re-run after experiments complete.')

## 5. Best Config for Phase 2

In [ ]:
if not results_df.empty:
    phase1_df = results_df[results_df['phase'] == 1]

    print('Recommended Phase 2 settings (best Phase 1 config per model):')
    print(f'{"="*65}')

    for model in MODELS:
        model_df = phase1_df[phase1_df['model'] == model]
        if model_df.empty:
            print(f'  {model:<20} — no results yet')
            continue

        best = model_df.loc[model_df['f1_macro'].idxmax()]
        print(f'  {model:<20} lr={best["lr"]}  bs={int(best["batch_size"])}  '
              f'F1={best["f1_macro"]:.4f}  AUC={best["auc"]:.4f}')

    print(f'{"="*65}')
    print('\nUpdate phase2_finetune.yaml for each model with these values.')
else:
    print('No phase 1 results yet.')

## 6. Per-Class Difficulty Analysis

In [ ]:
# load per-class reports if available
per_class_data = {}

if not results_df.empty:
    for _, row in best_per_model.iterrows():
        metrics_path = RESULTS_ROOT / row['model'] / row['run'] / 'test_metrics.json'
        if metrics_path.exists():
            with open(metrics_path) as f:
                data = json.load(f)
            if 'per_class_report' in data:
                per_class_data[row['model']] = data['per_class_report']

if per_class_data:
    # build comparison dataframe
    f1_by_class = {}
    for model, report in per_class_data.items():
        f1_by_class[model] = {
            cls: report[cls]['f1-score']
            for cls in class_names
            if cls in report
        }

    f1_df = pd.DataFrame(f1_by_class)
    f1_df['mean_f1'] = f1_df.mean(axis=1)
    f1_df = f1_df.sort_values('mean_f1')

    # heatmap
    fig, ax = plt.subplots(figsize=(10, 12))
    sns.heatmap(
        f1_df.drop(columns='mean_f1'),
        annot=True, fmt='.2f', cmap='RdYlGn',
        ax=ax, vmin=0, vmax=1
    )
    ax.set_title('Per-class F1 across models (sorted by mean F1, hardest first)')
    ax.set_xlabel('Model')
    ax.set_ylabel('Class')
    plt.tight_layout()
    plt.savefig('per_class_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: per_class_comparison.png')

    print(f'\n5 hardest classes (lowest mean F1 across models):')
    for cls, row in f1_df.head(5).iterrows():
        print(f'  {cls:<35} mean F1: {row["mean_f1"]:.3f}')
else:
    print('Per-class data not yet available. Re-run after experiments complete.')

## 7. Efficiency Comparison

The report requires a discussion of model efficiency — parameters, inference speed, and computational cost.

In [ ]:
import torch
from torchvision import models as tv_models
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_configs = {
    'resnet50':        lambda: tv_models.resnet50(weights=None),
    'efficientnet_b3': lambda: tv_models.efficientnet_b3(weights=None),
    'swin_t':          lambda: tv_models.swin_t(weights=None),
    'convnext_t':      lambda: tv_models.convnext_tiny(weights=None),
}

efficiency_records = []
dummy_input = torch.randn(1, 3, 224, 224).to(device)

for name, build_fn in model_configs.items():
    model = build_fn().to(device)
    model.eval()

    total_params = sum(p.numel() for p in model.parameters())

    # warmup
    with torch.no_grad():
        for _ in range(5):
            _ = model(dummy_input)

    # time 50 forward passes
    with torch.no_grad():
        t0 = time.time()
        for _ in range(50):
            _ = model(dummy_input)
        elapsed = (time.time() - t0) / 50 * 1000  # ms per image

    efficiency_records.append({
        'model':         name,
        'params_M':      round(total_params / 1e6, 1),
        'inference_ms':  round(elapsed, 2),
    })

    del model
    if device.type == 'cuda':
        torch.cuda.empty_cache()

eff_df = pd.DataFrame(efficiency_records)
print('Model efficiency comparison:')
print(eff_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

colours = ['#7570b3', '#1b9e77', '#d95f02', '#e7298a']

axes[0].bar(eff_df['model'], eff_df['params_M'], color=colours, edgecolor='white')
axes[0].set_title('Model size (parameters)')
axes[0].set_ylabel('Parameters (millions)')
axes[0].tick_params(axis='x', rotation=15)

axes[1].bar(eff_df['model'], eff_df['inference_ms'], color=colours, edgecolor='white')
axes[1].set_title('Inference speed (single image)')
axes[1].set_ylabel('ms per image')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle(f'Model Efficiency Comparison (device: {device})', y=1.02)
plt.tight_layout()
plt.savefig('model_efficiency.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: model_efficiency.png')

## 8. Summary for Report

This cell generates a clean summary table suitable for inclusion in the IEEE report.

In [ ]:
if not results_df.empty:
    # merge best results with efficiency data
    summary = best_per_model[['model','accuracy','f1_macro','precision','recall','auc','mAP']].copy()
    summary = summary.merge(eff_df[['model','params_M','inference_ms']], on='model', how='left')
    summary = summary.sort_values('f1_macro', ascending=False)

    print('Final summary table (for report):')
    print(summary.round(4).to_string(index=False))

    # save as CSV
    summary.to_csv('final_results_summary.csv', index=False)
    print('\nSaved: final_results_summary.csv')
else:
    print('No results yet — re-run after all experiments complete.')